Week 4: ETL Pipelines in Azure Databricks

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, sum, avg
import dlt

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("EcommerceDeltaLiveTable") \
    .getOrCreate()

# Task 1: Set up Delta Live Tables for real-time data ingestion and transformation

# Define the schema for the CSV data
schema = "product_id INT, customer_id INT, quantity INT, order_amount DECIMAL(10,2), order_date DATE"

# Ingest raw data into a Delta Live Table from the source (streaming CSV files)
@dlt.table(
  name="raw_orders",
  comment="Raw e-commerce order data ingested from files."
)
def raw_orders():
    return spark.read.format("csv").schema(schema).load("dbfs:/FileStore/order_fact.csv")

# Task 2: Data Processing with Delta Live Tables

# Clean the raw data to filter out invalid entries (e.g., orders with invalid order_amount)
@dlt.table(
  name="cleaned_orders",
  comment="Cleaned orders with valid order amounts."
)
def cleaned_orders():
    return dlt.read("raw_orders").filter(col("order_amount") > 0)

# Add new transformations to process and enhance the data

# 1. Calculate total revenue per product
@dlt.table(
  name="product_revenue",
  comment="Total revenue generated by each product."
)
def product_revenue():
    return dlt.read("cleaned_orders") \
              .groupBy("product_id") \
              .agg(round(sum(col("order_amount")), 2).alias("total_revenue"))

# 2. Calculate total quantity sold per product
@dlt.table(
  name="product_quantity_sold",
  comment="Total quantity sold per product."
)
def product_quantity_sold():
    return dlt.read("cleaned_orders") \
              .groupBy("product_id") \
              .agg(sum(col("quantity")).alias("total_quantity_sold"))

# 3. Calculate the average order value per product
@dlt.table(
  name="average_order_value",
  comment="Average order value per product."
)
def average_order_value():
    return dlt.read("cleaned_orders") \
              .groupBy("product_id") \
              .agg(round(avg(col("order_amount")), 2).alias("average_order_value"))

# 4. Calculate daily total sales (revenue) across all products
@dlt.table(
  name="daily_sales",
  comment="Daily total sales across all products."
)
def daily_sales():
    return dlt.read("cleaned_orders") \
              .groupBy("order_date") \
              .agg(round(sum(col("order_amount")), 2).alias("total_daily_sales"))

